<a href="https://colab.research.google.com/github/deckinhow/repositorio_grupo4/blob/develop/notebooks/02_Derick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Derick

## 0. Setup Inicial

Organizando o notebook para trabalho.

Etapas:
- Linkando o notebook com o repositório do GitHub;
- Importando as bibliotecas que serão utilizadas

In [1]:
!pip install gurobipy

In [2]:
!pip install PySCIPOpt

In [3]:
import gurobipy as gp
print(gp.gurobi.version())

(13, 0, 2)


In [4]:
import cvxpy as cp
print(cp.installed_solvers())

['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'GUROBI', 'HIGHS', 'OSQP', 'SCIP', 'SCIPY', 'SCS']


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
%%capture

import os

#Clonar o repositório do GitHub
if not os.path.exists('repositorio_grupo4/'):
  !git clone {'https://github.com/deckinhow/repositorio_grupo4.git'}
  %cd {'repositorio_grupo4/'}
else:
  %cd {'repositorio_grupo4/'}
  !git pull

#Bibliotecas que serão utilizadas
from dados_e_funcoes import funcoes #Funções criadas por nós
import statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import importlib
import yfinance
import cvxpy as cp
import gurobipy as gp

#Atualizar as funções salvas no arquivo de funções
importlib.reload(funcoes)

#Remover mensagens de aviso
warnings.filterwarnings('ignore')

In [7]:
ret_sp100 = pd.read_csv(
    '/content/drive/MyDrive/aaa/ret_sp100.csv',
    index_col=0,
    parse_dates=True
)

ret_ibov = pd.read_csv(
    '/content/drive/MyDrive/aaa/ret_ibov.csv',
    index_col=0,
    parse_dates=True
)

ret_idx_sp = pd.read_csv(
    '/content/drive/MyDrive/aaa/ret_idx_sp.csv',
    index_col=0,
    parse_dates=True
)

ret_idx_ib = pd.read_csv(
    '/content/drive/MyDrive/aaa/ret_idx_ib.csv',
    index_col=0,
    parse_dates=True
)


In [8]:
# IBOV

retornos_ativos = ret_ibov
retornos_indice = ret_idx_ib

In [9]:
retornos_ativos, retornos_indice = retornos_ativos.align(
    retornos_indice,
    join='inner',
    axis=0
)

In [10]:
retornos_ativos = retornos_ativos.fillna(0)

In [11]:
print(retornos_ativos.shape)
print(retornos_indice.shape)

(1736, 78)
(1736, 1)


## Modelo matemático de Index Tracking


**Formulação Matemática**
$$\min \frac{1}{T} \sum_{t=1}^{T} (\sum_{i \in I} w_i r_{t,i} - R_t)^2$$

Onde

$I$: Conjunto de ativos disponíveis

$T$: Número de períodos

$w_i$: Peso do ativo $i$ no portfólio de tracking

$z_i$: Varável binária para o ativo $i$

$R_t$: Rendimento do índice no período $t$

$r_t$: Rendimento do ativo $i$ no período $t$

$K$: O número máximo de ativos permitidos


**Restrições matemáticas:**
1. $\sum_{i \in I} w_i = 1$
2. $w_i \ge 0$
3. $w_i \le z_i$ e $\sum_{i \in I} z_i \le K$, onde $z_i \in \{0,1\}$


Veja que, a função matemática visa minimizar o Erro Quadrático Médio.

In [12]:
def otimizador_gurobi(retornos_ativos, retornos_indice, K): ##NAO TO USANDO O GUROBI OK POR CAUSA DA LICENÇA
    """
    Otimiza o portfólio rastreador limitando o número máximo de ativos.
    Utiliza o SCIP (Open-Source) para contornar limites de licença.
    """
    import cvxpy as cp

    n_ativos = retornos_ativos.shape[1]
    print("=== OTIMIZAÇÃO VIA SCIP (Alternativa Gratuita) ===")
    print(f"Ativos: {n_ativos}, K Máximo: {K}")

    # 1. Variáveis de decisão
    w = cp.Variable(n_ativos)
    y = cp.Variable(n_ativos, boolean=True)

    # 2. Criação do modelo matemático
    ret_portfolio = retornos_ativos.values @ w
    diferenca = ret_portfolio - retornos_indice.values.flatten()

    T = retornos_ativos.shape[0]
    min_eqm = cp.Minimize(cp.sum_squares(diferenca) / T)

    # 3. Restrições matemáticas do modelo
    restricoes = [
        cp.sum(w) == 1,
        w >= 0,
        cp.sum(y) <= K
    ]

    for i in range(n_ativos):
        restricoes.append(w[i] <= y[i])

    # 4. O Modelo Final
    modelo = cp.Problem(min_eqm, restricoes)

    # 5. Resolvendo com SCIP (Não tem limites de variáveis)
    modelo.solve(solver=cp.SCIP, verbose=False)

    # 6. Limpeza e Saída
    if w.value is None:
        raise ValueError("O solver SCIP não encontrou uma solução válida (w.value é None).")

    pesos_limpos = np.where(w.value < 1e-4, 0, w.value)
    pesos_finais = pesos_limpos / np.sum(pesos_limpos)

    return pesos_finais

In [13]:
from sklearn.linear_model import Lasso
import numpy as np

def otimizador_lasso(retornos_ativos, retornos_indice, alpha=0.001):
    """
    Otimiza a carteira usando Regressão Lasso (Linear).
    O parâmetro 'alpha': quanto maior o valor, mais ações
    terão seu peso zerado, ajudando a manter a carteira reduzida.
    """
    # Configuração:
    # positive=True impede pesos negativos (respeitando a regra de não operar vendido)
    # fit_intercept=False força o modelo a explicar os retornos puramente pelas ações
    lasso_model = Lasso(alpha=alpha, positive=True, fit_intercept=False, max_iter=10000)

    # Treinando o modelo (Tentando prever o índice usando as ações)
    lasso_model.fit(retornos_ativos, retornos_indice.values.flatten())

    # Extraindo os pesos que o modelo calculou
    pesos_crus = lasso_model.coef_

    # Normalização de Segurança:
    # A regressão não sabe que a soma precisa ser 100% (1.0). Nós forçamos isso aqui:
    soma = np.sum(pesos_crus)

    if soma > 0:
        pesos_finais = pesos_crus / soma
    else:
        # Fallback: Se o alpha for tão alto que ele zerou TODAS as ações, dividimos igual.
        n_ativos = retornos_ativos.shape[1]
        pesos_finais = np.ones(n_ativos) / n_ativos

    return pesos_finais

In [14]:
import xgboost as xgb

def otimizador_xgboost(retornos_ativos, K):
    """
    Otimiza a carteira usando Machine Learning (XGBoost).
    Treina o modelo para prever o retorno seguinte baseado no retorno anterior.
    Escolhe os K ativos com maior previsão de retorno.
    """
    n_ativos = retornos_ativos.shape[1]

    # 1. Preparando os dados para o formato que o XGBoost exige (X e y)
    # Vamos usar uma abordagem autorregressiva simples: o retorno de D-1 prevê o retorno de D.
    X_treino = retornos_ativos.iloc[:-1].values.flatten().reshape(-1, 1) # O Passado
    y_treino = retornos_ativos.iloc[1:].values.flatten()                 # O Futuro (Target)

    # 2. Configurando e Treinando o XGBoost
    # Usamos o Regressor pois estamos tentando prever um número contínuo (o retorno %)
    modelo_xgb = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=50,      # Número de árvores (50 é rápido e eficiente para esse volume)
        max_depth=3,          # Profundidade da árvore (evita overfitting)
        learning_rate=0.1,
        random_state=42
    )

    # A mágica do aprendizado acontece aqui
    modelo_xgb.fit(X_treino, y_treino)

    # 3. Prevendo o Futuro
    # Pegamos o último dia conhecido da janela de treino para prever o que vai acontecer no teste
    ultimos_dados_conhecidos = retornos_ativos.iloc[-1].values.reshape(-1, 1)
    previsoes_futuras = modelo_xgb.predict(ultimos_dados_conhecidos)

    # 4. Tomada de Decisão (A Estratégia)
    # Selecionamos os índices das K ações com as MAIORES previsões de retorno positivo
    top_k_indices = np.argsort(previsoes_futuras)[-K:]

    # 5. Montagem da Carteira
    # Zeramos a carteira e damos peso igual (1/K) apenas para as "vencedoras" previstas pelo ML
    pesos_finais = np.zeros(n_ativos)

    # Verificação de segurança: se o XGBoost prever que todas vão cair, investimos mesmo assim nas "menos piores"
    pesos_finais[top_k_indices] = 1.0 / K

    return pesos_finais

In [15]:
dias_por_ano = 252
tamanho_treino = dias_por_ano * 2

ativos_treino = retornos_ativos.iloc[:tamanho_treino]
bench_treino = retornos_indice.iloc[:tamanho_treino]

In [ ]:
# Configuração do Backtest
tamanho_teste = dias_por_ano * 1
numero_de_janelas = 5
K_maximo = 15 # Ks são arbitrários

resultados_metricas = []

retornos_oos_gurobi = pd.Series(dtype=float)
retornos_oos_lasso = pd.Series(dtype=float)
retornos_oos_xgb = pd.Series(dtype=float)

for i in range(numero_de_janelas):
    inicio_treino = i * tamanho_teste
    fim_treino = inicio_treino + tamanho_treino
    fim_teste = fim_treino + tamanho_teste

    # 1. Separação inicial (como você já fazia)
    ativos_treino_bruto = retornos_ativos.iloc[inicio_treino:fim_treino]
    bench_treino = retornos_indice.iloc[inicio_treino:fim_treino]
    ativos_teste_bruto = retornos_ativos.iloc[fim_treino:fim_teste]
    bench_teste = retornos_indice.iloc[fim_treino:fim_teste]

    # 2. OPÇÃO 2: Filtrando ativos sem nenhum NaN no treino
    # .dropna(axis=1) remove colunas que tenham qualquer NaN
    ativos_treino = ativos_treino_bruto.dropna(axis=1)

    # Crucial: Alinha a base de teste para ter APENAS os mesmos ativos do treino
    ativos_teste = ativos_teste_bruto[ativos_treino.columns]

    # Diagnóstico (Modificado para ver quantos ativos restaram)
    print(f"\n===== Janela {i} =====")
    print(f"Ativos originais: {ativos_treino_bruto.shape[1]} -> Ativos restantes sem NaNs: {ativos_treino.shape[1]}")
    print("NaNs restantes no treino:", ativos_treino.isna().sum().sum())
    print("NaNs índice treino:", bench_treino.isna().sum().sum())

    # Otimizando Pesos
    pesos_ideais = otimizador_gurobi(ativos_treino, bench_treino, K_maximo)
    pesos_ideais_lasso = otimizador_lasso(ativos_treino, bench_treino, alpha=0.001)
    pesos_xgb = otimizador_xgboost(ativos_treino, K_maximo)

    # Calculando Performance Relativa
    retornos_carteira_gurobi = ativos_teste @ pesos_ideais
    retornos_carteira_lasso = ativos_teste @ pesos_ideais_lasso
    retornos_carteira_xgb = ativos_teste @ pesos_xgb

    retornos_oos_gurobi = pd.concat([retornos_oos_gurobi, retornos_carteira_gurobi])
    retornos_oos_lasso = pd.concat([retornos_oos_lasso, retornos_carteira_lasso])
    retornos_oos_xgb = pd.concat([retornos_oos_xgb, retornos_carteira_xgb])

    bench_array = bench_teste.values.flatten()

    te_gurobi = np.std(retornos_carteira_gurobi - bench_array) * np.sqrt(dias_por_ano)
    te_lasso = np.std(retornos_carteira_lasso - bench_array) * np.sqrt(dias_por_ano)
    te_xgb = np.std(retornos_carteira_xgb - bench_array) * np.sqrt(dias_por_ano)

    resultados_metricas.append({
        'Janela de Teste': i + 1,
        'TE Gurobi (%)': round(te_gurobi * 100, 2),
        'TE Lasso (%)': round(te_lasso * 100, 2),
        'TE XGBoost (%)': round(te_xgb * 100, 2)
    })

df_metricas = pd.DataFrame(resultados_metricas)
display(df_metricas)


===== Janela 0 =====
Ativos originais: 78 -> Ativos restantes sem NaNs: 78
NaNs restantes no treino: 0
NaNs índice treino: 0
=== OTIMIZAÇÃO VIA SCIP (Alternativa Gratuita) ===
Ativos: 78, K Máximo: 15
